In [1]:
using Pkg
Pkg.activate("../test")

  Activating project at `~/repos/WaveGreen2D/test`


In [2]:
using Test
using StaticArrays

using WaveGreen2D.Chebyshev
using WaveGreen2D.Chebyshev: normalize, contains, clenshaw, gradient_clenshaw, hessian_clenshaw

In [3]:
@testset "0-D Chebyshev series" begin
    coefs = Array{Float64, 0}(undef)
    cs = ChebyshevSeries(coefs, SVector{0}(), SVector{0}())
    
    @test cs == coefs[]
end

Test Summary:        | Pass  Total  Time
0-D Chebyshev series |    1      1  0.3s


Test.DefaultTestSet("0-D Chebyshev series", Any[], 1, false, false, true, 1.778901908442778e9, 1.778901908787876e9, false, "In[3]", Random.Xoshiro(0xf90ebee62592e89c, 0xecd1f582601e60ef, 0x589c4c71e15f0f00, 0x85de0af3374b97fb, 0x23a46a4d2fda7e46))

In [5]:
@testset "1-D polynomial function" begin
    coefs = collect(UnitRange(1.0, 6.0))
    cs = ChebyshevSeries(coefs, SA[-1.0], SA[1.0])
    @test clenshaw(cs, 0.5) == -3
    @test gradient_clenshaw(cs, 0.5) == (-3, -42)
    @test hessian_clenshaw(cs, 0.5) == (-3, -42, -20)
end

Test Summary:           | Pass  Total  Time
1-D polynomial function |    3      3  0.7s


Test.DefaultTestSet("1-D polynomial function", Any[], 3, false, false, true, 1.778901921209029e9, 1.778901921947645e9, false, "In[5]", Random.Xoshiro(0xf90ebee62592e89c, 0xecd1f582601e60ef, 0x589c4c71e15f0f00, 0x85de0af3374b97fb, 0x23a46a4d2fda7e46))

In [22]:
@testset "2-D polynomial function" begin
    coefs = Float64[3*i*(j+1) for i in 1:6, j in 1:5]
    cs = ChebyshevSeries(coefs, SA[-1.0, -1.0], SA[1.0, 1.0])
    @test clenshaw(cs, SA[-0.5, 0.5]) == 58.5
    @test gradient_clenshaw(cs, SA[-0.5, 0.5]) == (58.5, 273, 117)
    @test hessian_clenshaw(cs, SA[-0.5, 0.5]) == (58.5, 273, -2418, 117, 546, -1116)
end

Test Summary:           | Pass  Total  Time
2-D polynomial function |    3      3  0.0s


Test.DefaultTestSet("2-D polynomial function", Any[], 3, false, false, true, 1.778888225132054e9, 1.778888225132132e9, false, "In[22]", Random.Xoshiro(0x1d5a7ff2ae995deb, 0xa972930d15ff0e9f, 0x85516b2b41753518, 0xc7162198873e663f, 0x25e43c1dea955ef9))

In [40]:
@testset "3-D polynomial function" begin
    coefs = Float64[0.25*(i-1)*(j+1)*(k+3) for i in 0:6, j in 0:4, k in 0:5]
    cs = ChebyshevSeries(coefs, SA[-1.0, -1.0, -1.0], SA[1.0, 1.0, 1.0])
    @test clenshaw(cs, SA[-0.5, 0.5, -0.5]) == 9
    @test gradient_clenshaw(cs, SA[-0.5, 0.5, -0.5]) == (9, -45, 18, 54)
    @test hessian_clenshaw(cs, SA[-0.5, 0.5, -0.5]) == (9, -45, -702, 18, -90, -150, 54, -270, 108, -492)
end

Test Summary:           | Pass  Total  Time
3-D polynomial function |    3      3  0.8s


Test.DefaultTestSet("3-D polynomial function", Any[], 3, false, false, true, 1.778899941647523e9, 1.778899942497476e9, false, "In[40]", Random.Xoshiro(0x1d5a7ff2ae995deb, 0xa972930d15ff0e9f, 0x85516b2b41753518, 0xc7162198873e663f, 0x25e43c1dea955ef9))

In [15]:
@testset "1-D transcedental function" begin
    # Coefficients for f(x) = sin(x)
    coefs = [6.0219470125554653e-01, 5.1362516667910696e-01, -1.0354634426296377e-01, -1.3732034234358511e-02, 1.3586698380903617e-03, 1.0726309440600129e-04, -7.0462967934650502e-06, -3.9639025062457094e-07, 1.9499597787191747e-08, 8.5229287361514171e-10, -3.3516569268963781e-11, -1.1979172283756631e-12, 3.9247734881581288e-14, 1.1922754450388594e-15]
    cs = ChebyshevSeries(coefs, SA[0.0], SA[π/2])
    
    x = π/3
    
    @test cs(x) ≈ sin(x)
    
    y, dy_dx = gradient(cs, x)
    @test y ≈ sin(x)
    @test dy_dx ≈ cos(x)

    y, dy_dx, d²y_dx² = hessian(cs, x)
    @test y ≈ sin(x)
    @test dy_dx ≈ cos(x)
    @test d²y_dx² ≈ -sin(x)
end

Test Summary:              | Pass  Total  Time
1-D transcedental function |    6      6  0.3s


Test.DefaultTestSet("1-D transcedental function", Any[], 6, false, false, true, 1.77890270593723e9, 1.778902706187606e9, false, "In[15]", Random.Xoshiro(0xf90ebee62592e89c, 0xecd1f582601e60ef, 0x589c4c71e15f0f00, 0x85de0af3374b97fb, 0x23a46a4d2fda7e46))

In [12]:
@testset "1-D transcedental function" begin
    # Coefficients for f(x) = sin(x)
    coefs = [
        6.0219470125554653e-01, 5.1362516667910696e-01, -1.0354634426296377e-01,
        -1.3732034234358511e-02, 1.3586698380903617e-03, 1.0726309440600129e-04,
        -7.0462967934650502e-06, -3.9639025062457094e-07, 1.9499597787191747e-08,
        8.5229287361514171e-10, -3.3516569268963781e-11, -1.1979172283756631e-12,
        3.9247734881581288e-14, 1.1922754450388594e-15,
    ]
    cs = ChebyshevSeries(coefs, SA[0.0], SA[π/2])
    x = π / 3

    @test cs(x) ≈ sin(x)

    y, dy_dx = gradient(cs, x)
    @test y ≈ sin(x)
    @test dy_dx ≈ cos(x)

    y, dy_dx, d²y_dx² = hessian(cs, x)
    @test y ≈ sin(x)
    @test dy_dx ≈ cos(x)
    @test d²y_dx² ≈ -sin(x)
end

(0.8660254037844386, 0.4999999999999999)